In [1]:
%load_ext autoreload
%autoreload 2

In [16]:
# Copyright 2017 The TensorFlow Authors All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# =============================================================================


import tensorflow as tf
from tqdm import tqdm

from migration.models import vrnn
from pydantic import PositiveInt
from migration.config import TrainingConfig, DatasetConfig
import migration.datasets as datasets

from migration.models.vrnn_elbo import VRNN
from pathlib import Path

# get batch and model
def create_dataset(cfg: DatasetConfig, repeat : bool) -> tf.data.Dataset:

    return datasets.get_Tensorflow_AIS_dataset(
                    cfg.training_pickle,
                    cfg.batch_size,
                    cfg.encoding_bins.lat,
                    cfg.encoding_bins.lon, 
                    cfg.encoding_bins.sog,
                    cfg.encoding_bins.cog, 
                    shuffle=cfg.shuffle,
                    repeat=repeat)

def create_model(mean_path : Path, latent_size : PositiveInt, total_bins : PositiveInt):
    # Convert the mean of the training set to logit space so it can be used to
    # initialize the bias of the generative distribution.
    mean = datasets.get_AIS_dataset_mean(mean_path)
    generative_bias_init = -tf.math.log(1. / tf.clip_by_value(mean, 0.0001, 0.9999) - 1)
    generative_distribution_class = vrnn.ConditionalBernoulliDistribution
    model = VRNN(total_bins,
                             latent_size,
                             generative_distribution_class,
                             generative_bias_init=generative_bias_init,
                             raw_sigma_bias=0.5, num_samples=1)
    return model



def run_train(cfg : TrainingConfig):

    if cfg.random_seed:
        tf.random.set_seed(cfg.random_seed)

    dataset : tf.data.Dataset = create_dataset(cfg.dataset, repeat=True)    
    model = create_model(cfg.dataset.mean_pickle, cfg.model.latent_size, cfg.dataset.encoding_bins.total)
    optimizer = tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate)
    
    @tf.function
    def train_step(x,y, lengths):
        with tf.GradientTape() as tape:
            bound = model((x, y),lengths)
            # Compute lower bounds on the log likelihood.
            bound = tf.reduce_mean(input_tensor=bound / tf.cast(lengths, dtype=tf.float32))
            loss = -bound
        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        return loss
    
    for _ in tqdm(range(cfg.epochs)):
        for inputs, targets, lengths in dataset:
            print(train_step(inputs, targets, lengths))


In [17]:
from migration.config import TrainingConfig, DatasetConfig

cfg = TrainingConfig(
    dataset=DatasetConfig(
        training_pickle='../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl',
        mean_pickle='../../data/ct_2017010203_10_20/mean.pkl'
    ))

In [18]:
from migration.config import TrainingConfig, DatasetConfig

cfg = TrainingConfig(
    dataset=DatasetConfig(
        training_pickle='../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl',
        mean_pickle='../../data/ct_2017010203_10_20/mean.pkl'
    ))
# fh = logging.FileHandler(os.path.join(config.logdir,config.log_filename+".log"))
# # get TF logger
# logger = logging.getLogger('tensorflow')
# logger.addHandler(fh)
run_train(cfg)


  0%|          | 0/5 [00:00<?, ?it/s]

tf.Tensor(21.18817, shape=(), dtype=float32)
tf.Tensor(21.040554, shape=(), dtype=float32)
tf.Tensor(21.009237, shape=(), dtype=float32)
tf.Tensor(21.268301, shape=(), dtype=float32)
tf.Tensor(21.769032, shape=(), dtype=float32)
tf.Tensor(21.679272, shape=(), dtype=float32)
tf.Tensor(21.149681, shape=(), dtype=float32)
tf.Tensor(21.404388, shape=(), dtype=float32)
tf.Tensor(20.57955, shape=(), dtype=float32)
tf.Tensor(20.823666, shape=(), dtype=float32)
tf.Tensor(20.689106, shape=(), dtype=float32)
tf.Tensor(20.277466, shape=(), dtype=float32)
tf.Tensor(20.547245, shape=(), dtype=float32)
tf.Tensor(20.370098, shape=(), dtype=float32)
tf.Tensor(21.159056, shape=(), dtype=float32)
tf.Tensor(20.64988, shape=(), dtype=float32)


  0%|          | 0/5 [01:07<?, ?it/s]


KeyboardInterrupt: 